In [ ]:
pip install jieba

In [4]:
import re, requests, codecs, time, random, jieba,tushare
import jieba.analyse
from lxml import html

# proxies={"http" : "123.53.86.133:61234"}
proxies = None
headers = {
    'Host': 'guba.eastmoney.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/49.0.2623.221 Safari/537.36 SE 2.X MetaSr 1.0'}

def get_url(stocknum,page):
    url = 'http://guba.eastmoney.com/list,' + str(stocknum) + '_' + str(page) + '.html'
    try:
        text = requests.get(url, headers=headers, proxies=proxies, timeout=20)
        requests.adapters.DEFAULT_RETRIES = 5
        s = requests.session()
        s.keep_alive = False
        text = html.fromstring(text.text)
        urls = text.xpath('//div[@id="articlelistnew"]//div[@class="articleh normal_post"]/span[3]/a/@href')
        # print(urls)
    except Exception as e:
        time.sleep(random.random() + random.randint(1, 3))
        urls = ''
    return urls


def get_comments(urls):
    for newurl in urls:
        newurl1 = 'http://guba.eastmoney.com' + newurl
        # print(newurl1)
        try:
            text1 = requests.get(newurl1, headers=headers, proxies=proxies, timeout=20)
            requests.adapters.DEFAULT_RETRIES = 5
            s = requests.session()
            s.keep_alive = False
            text1 = html.fromstring(text1.text)
            times1 = text1.xpath('//div[@class="zwfbtime"]/text()|//div[@class="zwli clearfix"]/div[4]/div/div[2]/text()')
            times = '!'.join(re.sub(re.compile('发表于| '), '', x)[:10] for x in times1).split('!')
            # print(times)
            # times=list(map(lambda x:re.sub(re.compile('发表于| '),'',x)[:10],times))
            comments1 = text1.xpath('//div[@class="stockcodec .xeditor"]/text()|//div[@class="zwli clearfix"]/div[4]/div/div[3]/div/text()')
            comments = '!'.join(w.strip() for w in comments1).split('!')
            if comments == ['']:
                continue
            else:
                dic = dict(zip(times, comments))
                save_to_file(dic)
        except:
            print('error!!!!')
            time.sleep(random.random() + random.randint(0, 3))

    # if times and comments:
        # dic.append({'time':times,'comment':comments})
    # return dic


def save_to_file(dic):
    if dic:
        # dic=dic
        # print(dic)
        # df=pd.DataFrame([dic]).T
        # df.to_excel('eastnoney.xlsx')
        for i, j in dic.items():
            output = '{},{}\n'.format(i, j)
            datacsv = stocknum + '-' + 'eastmoney.csv'
            f = codecs.open(datacsv, 'a+', 'utf-8')
            f.write(output)
            f.close()

def jb_file(stocknum):
    address = stocknum + '-' + 'eastmoney.csv'
    with open(address, 'r', encoding='utf-8') as f:
        comment_list = []
        for comments in f.readlines():
            comment_list.extend(comments.split(',')[1:])
        comment_str = ''.join(comment_list)

    excludes = ['满仓','空仓',"涨", "跌", "看好", "积极", "垃圾", "看跌", '卖', '买', '看涨', '利好', "利空", '庄', '诱多', '诱空', '出货', '底', '顶','跳水','拉升']

    kwords = jieba.analyse.textrank(comment_str, topK=50, withWeight=True, allowPOS=('ns', 'n', 'vn', 'v'))
    print('关键词统计')
    for kword in kwords[:50]:
        print("{0:{2:}<6}{1:>6.4}".format(kword[0], kword[1], chr(12288)))

    words = jieba.lcut(comment_str)
    counts = {}
    for word in words:
        if word in excludes:
            counts[word] = counts.get(word, 0) + 1
    items = list(counts.items())
    items.sort(key=lambda x: x[1], reverse=True)
    print('\n\n交易情绪')
    for i in range(len(items)):
        word, count = items[i]
        print("{0:{2:}<6}{1:>3}".format(word, count, chr(12288)))

def get_stockcode():
    all_stock = tushare.get_stock_basics()
    all_stock_list = all_stock.index.tolist()
    while True:
        s_code = input("输入股票代码：")
        if s_code in all_stock_list:
            pass
            return s_code
        else:
            print('无法找到该股票，请重新')
            continue

def get_page():
    while True:
        g_page = input('需要爬取的页数:')
        if g_page.isdigit() and int(g_page) > 0:
            pass
            return int(g_page)
        else:
            print('输入需为数字且大于零，请重新')
            continue

if __name__ == '__main__':
    stocknum = get_stockcode()
    g_page = get_page()
    for page in range(0, int(g_page)):
        print('正在爬取第{}页'.format(page))
        urls = get_url(stocknum,page)
        dic = get_comments(urls)
    jb_file(stocknum)


本接口即将停止更新，请尽快使用Pro版接口：https://tushare.pro/document/2


URLError: <urlopen error [Errno 11001] getaddrinfo failed>